In [ ]:
import os
import pandas as pd
import argparse
import re
import lxml.etree as ET
from xml_ns import ns

import vocab_maps
from collections import defaultdict

In [ ]:
from foundry.transforms import Dataset

ccda_documents = Dataset.get("ccda_documents")
ccda_documents_files = ccda_documents.files().download()

In [ ]:
# Define namespaces to use when searching
ns = {'hl7': 'urn:hl7-org:v3'}

# Directory containing the XML files
directory_path = './resources/'

# Initialize a list to store records for DataFrame
data_records = []

In [ ]:
#with open(ccda_documents_files['CCD-Sample.xml'], 'rb') as file:
#    tree = ET.parse(file)
#    print(ET.tostring(tree, pretty_print=True).decode("utf-8"))

In [ ]:
#with open(ccda_documents_files['CCD-Sample.xml'], 'rb') as file:
#    tree = ET.parse(file)
#    # Load and parse the XML file from the dataset
#    try:
#        with open(file_path, 'rb') as file:
#            tree = ET.parse(file)
#
#        # Define elements to extract
#        elements = [
#            ('administrativeGenderCode', 'Gender'),
#            ('raceCode', 'Race'),
#            ('ethnicGroupCode', 'Ethnicity'),
#            ('birthTime', 'BirthTime')
#        ]
#
#        # Extract data for each element
#        record = {'Filename': filename}
#        for tag, label in elements:
#            element = tree.find(f'.//hl7:{tag}', namespaces=ns)
#            value = element.get('displayName') if tag != 'birthTime' and element is not None else (
#                element.get('value') if element is not None else 'Unknown'
#            )
#            record[label] = value
#
#        # Append data to the records list
#        data_records.append(record)
#
#    except Exception as e:
#        print(f"Failed to parse {filename}: {e}")
#
## Create DataFrame from the records
#ccda_snooper_people = pd.DataFrame(data_records)
#
## Print the DataFrame
#print(ccda_snooper_people)
#
## Optionally, save the DataFrame to a CSV file
##df.to_csv('output_summary_with_loop.csv', index=False)

In [ ]:
# Iterate over all XML files in the directory, fetch needed attributes from each, and build a dataset of attributes per file.
for filename in os.listdir(directory_path):
    print(filename)
    if filename.endswith('.xml'):
        file_path = os.path.join(directory_path, filename)

        # Load and parse the XML file
        try:
            with open(file_path, 'rb') as file:
                tree = ET.parse(file)

            # Define elements to extract
            elements = [
                ('administrativeGenderCode', 'Gender'),
                ('raceCode', 'Race'),
                ('ethnicGroupCode', 'Ethnicity'),
                ('birthTime', 'BirthTime')
            ]
            
            # Extract data for each element
            record = {'Filename': filename}
            for tag, label in elements:
                element = tree.find(f'.//hl7:{tag}', namespaces=ns)
                value = element.get('displayName') if tag != 'birthTime' and element is not None else (
                    element.get('value') if element is not None else 'Unknown'
                )
                record[label] = value

            # Append data to the records list
            data_records.append(record)

        except Exception as e:
            print(f"Failed to parse {filename}: {e}")

# Create DataFrame from the records
ccda_snooper_people = pd.DataFrame(data_records)

# Print the DataFrame
print(ccda_snooper_people)

# Optionally, save the DataFrame to a CSV file
#df.to_csv('output_summary_with_loop.csv', index=False)


In [ ]:
# Save dataset to HDFS/Spark in Foundry

from foundry.transforms import Dataset


dq_ccda_snooper_people = Dataset.get("dq_ccda_snooper_people")
dq_ccda_snooper_people.write_table(ccda_snooper_people)
print(f"wrote dq_ccda_snooper_people dataset")

In [ ]:

def code_system_snooper():

    # Initialize a list to store records
    data_records = []

    # Iterate over all XML files in the directory
    for filename in os.listdir(directory_path):
        if filename.endswith('.xml'):
            file_path = os.path.join(directory_path, filename)

            # Load and parse the XML file
            try:
                with open(file_path, 'rb') as file:
                    tree = ET.parse(file)

                # Find elements with codeSystem='2.16.840.1.113883.6.96'
                elements_with_code_system = tree.findall(".//*[@codeSystem='2.16.840.1.113883.6.96']", namespaces=ns)

                # If such elements are found, extract relevant information
                if elements_with_code_system:
                    for element in elements_with_code_system:
                        record = {'Filename': filename}

                        # Extract common attributes from the element (e.g., displayName, code, etc.)
                        record['Code'] = element.get('code', 'Unknown')
                        record['DisplayName'] = element.get('displayName', 'Unknown')
                        record['CodeSystem'] = element.get('codeSystem', 'Unknown')

                        # Append data to the records list
                        data_records.append(record)
                else:
                    print(f"No elements with codeSystem='2.16.840.1.113883.6.96' found in {filename}")

            except Exception as e:
                print(f"Failed to parse {filename}: {e}")

    # Create DataFrame from the records
    ccda_snooper_code_system = pd.DataFrame(data_records)

    # Print the DataFrame
    print(ccda_snooper_code_system)
    
    return ccda_snooper_code_system

# Optionally, save the DataFrame to a CSV file
# ccda_snooper_code_system.to_csv('output_summary_with_code_system.csv', index=False)


In [ ]:

## SNOOP SECTION

# Section Snooper, but no dataset output???
# execution commented out below.

In [ ]:
ccda_code_values_columns = [ 'filename', 'section','section_code','section_name', 'codeSystem', 'code',
                             'value_type', 'value_unit',
                             'value_value', 'value_code', 'value_codeSystem',  'value_text',
                             'path']
def snoop_section(tree, filename):
    trace_df = pd.DataFrame(columns=ccda_code_values_columns)
    section_elements = tree.xpath(".//section")
    #print(ET.tostring(tree, pretty_print=True).decode("utf-8"))
    for section_element in section_elements:

        section_template_id = "n/a"
        section_template_ele = section_element.findall("templateId",ns)
        section_code = "n/a"
        section_code_ele = section_element.findall("code",ns)
        section_name = "n/a"
        if len(section_template_ele) > 0:
            section_template_id = section_template_ele[0].get('root')
            #section_code = section_code_ele[0].get('code')
            #section_name = section_code_ele[0].get('displayName')
        entry_elements = section_element.findall("entry", ns)
        for entry_ele in entry_elements:
            print(entry_ele)
            value_dict = defaultdict(list)  # keys are paths
            value_elements = entry_ele.findall(".//value", ns)
            for value_ele in value_elements:
                value_path = re.sub(r'{.*?}', '', tree.getelementpath(value_ele))
                value_path = "/".join(value_path.split("/")[:-1])
                value_attribs_dict = { re.sub(r'{.*}', '', a):
                                       re.sub(r'{.*}', '', value_ele.attrib[a])
                                       for a in value_ele.attrib  }
                value_dict[value_path].append( (value_attribs_dict, value_ele.text) )

            code_elements = entry_ele.findall(".//code", ns)
            for code_ele in code_elements:
                code_path = re.sub(r'{.*?}', '', tree.getelementpath(code_ele))
                code_path = "/".join(code_path.split("/")[:-1])
                value_tuple_list = value_dict[code_path] # tuple is (dict, text)
                for value_tuple in value_tuple_list:
                    new_row = pd.DataFrame([{
                        'filename': filename,
                        'section': section_template_id,
                        'section_code': section_code,
                        'section_name': section_name,
                        'path': code_path,
                        'code': code_ele.get('code'),
                        'codeSystem': code_ele.get('codeSystem'),
                        'value_type': value_tuple[0].get('type', ''),
                        'value_unit': value_tuple[0].get('unit', ''),
                        'value_value': value_tuple[0].get('value', ''),
                        'value_code': value_tuple[0].get('code', ''),
                        'value_codeSystem': value_tuple[0].get('codeSystem', ''),
                        'value_text': value_tuple[1].strip() if value_tuple[1] else ''
                    }])
                    trace_df = pd.concat([trace_df, new_row], ignore_index=True)
    return(trace_df)

In [ ]:
#all_files_df = pd.DataFrame(columns=ccda_code_values_columns)
## Iterate over all XML files in the directory
#for filename in os.listdir(directory_path):
#    if filename.endswith('.xml'):
#        file_path = os.path.join(directory_path, filename)
#        print(file_path)
#        tree = ET.parse(file_path)
#        file_df = snoop_section(tree, filename)
#        print(file_df)
#        #all_files_df = pd.concat([all_files_df, file_df], ignore_index=True)
#        #print(all_files_df)
#        #all_files_df.to_csv(f"raw_section_snooper_Adam2.csv",sep=",", header=True, index=False)
